In [1]:
import os

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("metricas-viagens")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")   # timestamps do lakehouse estão em UTC
    .config("spark.sql.shuffle.partitions", "8")    # dataset pequeno -> menos partições
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## 1. Leitura das tabelas de origem

In [2]:
LAKEHOUSE = "/home/jovyan/lakehouse"

def ler_staging(tabela: str):
    return spark.read.parquet(f"{LAKEHOUSE}/staging/{tabela}")

viagens    = ler_staging("viagens")
veiculos   = ler_staging("veiculos")
motoristas = ler_staging("motoristas")
geocercas  = ler_staging("geocercas")
posicoes   = ler_staging("posicoes")

for nome, df in [("viagens", viagens), ("veiculos", veiculos),
                 ("motoristas", motoristas), ("geocercas", geocercas),
                 ("posicoes", posicoes)]:
    print(f"{nome:11s}: {df.count():>6} linhas x {len(df.columns):>2} colunas")

viagens    :   2852 linhas x 18 colunas
veiculos   :    142 linhas x 17 colunas


motoristas :    111 linhas x 16 colunas
geocercas  :     38 linhas x 13 colunas


posicoes   :  54015 linhas x 15 colunas


## 2. Tabela consolidada — `viagens_enriquecidas`

In [3]:
# --- 2.1 Telemetria agregada por viagem (a partir do GPS) ---
metricas_pos = (
    posicoes.groupBy("viagem_id")
    .agg(
        F.round(F.avg("velocidade_kmh"), 2).alias("velocidade_media_kmh"),
        F.count("*").alias("num_posicoes"),
        F.min("timestamp").alias("primeira_posicao_ts"),
        F.max("timestamp").alias("ultima_posicao_ts"),
    )
)
metricas_pos.show(5, truncate=False)

+----------+--------------------+------------+--------------------------+--------------------------+
|viagem_id |velocidade_media_kmh|num_posicoes|primeira_posicao_ts       |ultima_posicao_ts         |
+----------+--------------------+------------+--------------------------+--------------------------+
|VIA-002674|48.48               |27          |2026-04-28 21:37:55.188432|2026-04-29 13:22:55.188432|
|VIA-002529|58.0                |27          |2026-04-28 21:35:35.483511|2026-04-29 13:11:35.483511|
|VIA-001792|51.67               |30          |2026-04-28 21:37:26.315696|2026-04-29 15:59:26.315696|
|VIA-002227|59.74               |23          |2026-04-28 21:38:50.418646|2026-04-29 11:34:50.418646|
|VIA-000899|51.69               |16          |2026-04-28 21:35:44.30146 |2026-04-29 06:55:44.30146 |
+----------+--------------------+------------+--------------------------+--------------------------+
only showing top 5 rows


In [4]:
veic = veiculos.select(
    "veiculo_id",
    F.col("placa").alias("veiculo_placa"),
    F.col("marca").alias("veiculo_marca"),
    F.col("modelo").alias("veiculo_modelo"),
    F.col("tipo").alias("veiculo_tipo"),
    F.col("capacidade_kg").alias("veiculo_capacidade_kg"),
    F.col("status").alias("veiculo_status"),
)
mot = motoristas.select(
    "motorista_id",
    F.col("nome").alias("motorista_nome"),
    F.col("categoria_cnh").alias("motorista_categoria_cnh"),
    F.col("base_operacional").alias("motorista_base_operacional"),
    F.col("status").alias("motorista_status"),
)
geo_orig = geocercas.select(
    F.col("geocerca_id").alias("geocerca_origem_id"),
    F.col("nome").alias("origem_nome"),
    F.col("tipo").alias("origem_tipo"),
    F.col("uf").alias("origem_uf"),
)
geo_dest = geocercas.select(
    F.col("geocerca_id").alias("geocerca_destino_id"),
    F.col("nome").alias("destino_nome"),
    F.col("tipo").alias("destino_tipo"),
    F.col("uf").alias("destino_uf"),
)

def horas(fim, ini):
    return F.round((F.col(fim).cast("long") - F.col(ini).cast("long")) / 3600.0, 2)

In [5]:
# --- 2.3 Montagem da tabela consolidada ---
viagens_enriquecidas = (
    viagens
    .join(veic,     "veiculo_id",          "left")
    .join(mot,      "motorista_id",         "left")
    .join(geo_orig, "geocerca_origem_id",   "left")
    .join(geo_dest, "geocerca_destino_id",  "left")
    .join(metricas_pos, "viagem_id",        "left")
    # métricas temporais
    .withColumn("duracao_horas",          horas("data_fim_real",     "data_inicio"))
    .withColumn("duracao_prevista_horas", horas("data_fim_prevista", "data_inicio"))
    .withColumn("atraso_horas",
                F.round(F.col("duracao_horas") - F.col("duracao_prevista_horas"), 2))
    .withColumn("atrasada_flag",
                (F.col("status") == "ATRASADA")
                | (F.col("data_fim_real").isNotNull()
                   & (F.col("data_fim_real") > F.col("data_fim_prevista"))))
    .withColumn("mes", F.date_format("data_inicio", "yyyy-MM"))
)

# ordem de colunas legível
colunas = [
    "viagem_id", "veiculo_id", "motorista_id",
    "geocerca_origem_id", "geocerca_destino_id",
    "data_inicio", "data_fim_prevista", "data_fim_real",
    "status", "distancia_km", "peso_carga_kg", "nota_fiscal",
    "veiculo_placa", "veiculo_marca", "veiculo_modelo", "veiculo_tipo",
    "veiculo_capacidade_kg", "veiculo_status",
    "motorista_nome", "motorista_categoria_cnh",
    "motorista_base_operacional", "motorista_status",
    "origem_nome", "origem_tipo", "origem_uf",
    "destino_nome", "destino_tipo", "destino_uf",
    "velocidade_media_kmh", "num_posicoes",
    "primeira_posicao_ts", "ultima_posicao_ts",
    "duracao_horas", "duracao_prevista_horas", "atraso_horas",
    "atrasada_flag", "mes",
]
viagens_enriquecidas = viagens_enriquecidas.select(*colunas).cache()

print("viagens_enriquecidas:", viagens_enriquecidas.count(), "linhas x",
      len(viagens_enriquecidas.columns), "colunas")
viagens_enriquecidas.printSchema()

viagens_enriquecidas: 2852 linhas x 37 colunas
root
 |-- viagem_id: string (nullable = true)
 |-- veiculo_id: string (nullable = true)
 |-- motorista_id: string (nullable = true)
 |-- geocerca_origem_id: string (nullable = true)
 |-- geocerca_destino_id: string (nullable = true)
 |-- data_inicio: timestamp (nullable = true)
 |-- data_fim_prevista: timestamp (nullable = true)
 |-- data_fim_real: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- distancia_km: integer (nullable = true)
 |-- peso_carga_kg: integer (nullable = true)
 |-- nota_fiscal: string (nullable = true)
 |-- veiculo_placa: string (nullable = true)
 |-- veiculo_marca: string (nullable = true)
 |-- veiculo_modelo: string (nullable = true)
 |-- veiculo_tipo: string (nullable = true)
 |-- veiculo_capacidade_kg: integer (nullable = true)
 |-- veiculo_status: string (nullable = true)
 |-- motorista_nome: string (nullable = true)
 |-- motorista_categoria_cnh: string (nullable = true)
 |-- motorista_base_o

###  Viagens por mês e por status


In [6]:
viagens_por_mes_status = (
    viagens_enriquecidas
    .groupBy("mes", "status")
    .agg(F.count("*").alias("total_viagens"))
    .orderBy("mes", F.desc("total_viagens"))
)
viagens_por_mes_status.show(truncate=False)

+-------+-----------+-------------+
|mes    |status     |total_viagens|
+-------+-----------+-------------+
|2026-04|CONCLUIDA  |1876         |
|2026-04|EM_TRANSITO|423          |
|2026-04|ATRASADA   |312          |
|2026-04|CANCELADA  |241          |
+-------+-----------+-------------+



### Tempo médio de viagem por rota (origem → destino)


In [7]:
tempo_medio_por_rota = (
    viagens_enriquecidas
    .filter(F.col("duracao_horas").isNotNull())
    .groupBy("geocerca_origem_id", "geocerca_destino_id", "origem_nome", "destino_nome")
    .agg(
        F.round(F.avg("duracao_horas"), 2).alias("tempo_medio_horas"),
        F.count("*").alias("total_viagens"),
    )
    .orderBy(F.desc("total_viagens"), F.desc("tempo_medio_horas"))
)
print("rotas distintas:", tempo_medio_por_rota.count())
tempo_medio_por_rota.show(15, truncate=38)

rotas distintas: 215


+------------------+-------------------+-----------------------------------+----------------------------+-----------------+-------------+
|geocerca_origem_id|geocerca_destino_id|                        origem_nome|                destino_nome|tempo_medio_horas|total_viagens|
+------------------+-------------------+-----------------------------------+----------------------------+-----------------+-------------+
|          GEO-0001|           GEO-0004|           CD São Paulo - Guarulhos|CD Belo Horizonte - Contagem|            40.56|           18|
|          GEO-0002|           GEO-0037|                  CD Curitiba - CIC|     Cliente Polo Industrial|            39.44|           18|
|          GEO-0002|           GEO-0036|                  CD Curitiba - CIC|         Cliente Zona Franca|            48.71|           17|
|          GEO-0003|           GEO-0024|CD Rio de Janeiro - Duque de Caxias|          Cliente Automotivo|            46.12|           17|
|          GEO-0006|           GEO

### Velocidade média por viagem

In [8]:
velocidade_media_por_viagem = (
    viagens_enriquecidas
    .select("viagem_id", "origem_nome", "destino_nome", "distancia_km",
            "num_posicoes", "velocidade_media_kmh")
    .filter(F.col("velocidade_media_kmh").isNotNull())
    .orderBy(F.desc("velocidade_media_kmh"))
)

com_tel = velocidade_media_por_viagem.count()
sem_tel = viagens_enriquecidas.filter(F.col("velocidade_media_kmh").isNull()).count()
print(f"viagens com telemetria: {com_tel} | sem posições GPS: {sem_tel}")

velocidade_media_por_viagem.show(10, truncate=28)

velocidade_media_por_viagem.select(
    F.round(F.avg("velocidade_media_kmh"), 2).alias("media_frota_kmh"),
    F.min("velocidade_media_kmh").alias("min_kmh"),
    F.max("velocidade_media_kmh").alias("max_kmh"),
).show()

viagens com telemetria: 2415 | sem posições GPS: 437


+----------+----------------------------+----------------------------+------------+------------+--------------------+
| viagem_id|                 origem_nome|                destino_nome|distancia_km|num_posicoes|velocidade_media_kmh|
+----------+----------------------------+----------------------------+------------+------------+--------------------+
|VIA-000042|           CD Curitiba - CIC|          Cliente Construção|        2117|          22|               79.05|
|VIA-001951|CD Belo Horizonte - Contagem|Cliente Distribuidora Beb...|        1400|          14|               78.07|
|VIA-000936|    CD São Paulo - Guarulhos|  Cliente Terminal Portuário|        2290|          15|               76.33|
|VIA-002361|    CD Porto Alegre - Canoas|        CD Recife - Jaboatão|        1370|          15|                75.6|
|VIA-001318|CD Rio de Janeiro - Duque...|     Cliente Polo Industrial|         986|          15|               75.47|
|VIA-000186|    CD São Paulo - Guarulhos|CD Rio de Janei

+---------------+-------+-------+
|media_frota_kmh|min_kmh|max_kmh|
+---------------+-------+-------+
|          55.08|  32.18|  79.05|
+---------------+-------+-------+



### Taxa de atraso por mês

In [9]:
taxa_atraso_mensal = (
    viagens_enriquecidas
    .groupBy("mes")
    .agg(
        F.sum(F.col("atrasada_flag").cast("int")).alias("viagens_atrasadas"),
        F.count("*").alias("viagens_total"),
    )
    .withColumn("taxa_atraso",
                F.round(F.col("viagens_atrasadas") / F.col("viagens_total"), 4))
    .orderBy("mes")
)
taxa_atraso_mensal.show(truncate=False)

+-------+-----------------+-------------+-----------+
|mes    |viagens_atrasadas|viagens_total|taxa_atraso|
+-------+-----------------+-------------+-----------+
|2026-04|1691             |2852         |0.5929     |
+-------+-----------------+-------------+-----------+



### Top 10 motoristas por viagens concluídas

In [10]:
ranking = Window.orderBy(F.desc("viagens_concluidas"), F.asc("motorista_id"))

top_motoristas = (
    viagens_enriquecidas
    .filter(F.col("status") == "CONCLUIDA")
    .groupBy("motorista_id", "motorista_nome")
    .agg(F.count("*").alias("viagens_concluidas"))
    .withColumn("posicao_rank", F.row_number().over(ranking))
    .filter(F.col("posicao_rank") <= 10)
    .select("posicao_rank", "motorista_id", "motorista_nome", "viagens_concluidas")
    .orderBy("posicao_rank")
)
top_motoristas.show(truncate=False)

+------------+------------+-------------------+------------------+
|posicao_rank|motorista_id|motorista_nome     |viagens_concluidas|
+------------+------------+-------------------+------------------+
|1           |MOT-0031    |SR. PAULO PINTO    |31                |
|2           |MOT-0055    |ANNA LIZ SIQUEIRA  |30                |
|3           |MOT-0045    |LORENZO LOPES      |29                |
|4           |MOT-0030    |DR. DIEGO PORTO    |27                |
|5           |MOT-0073    |CALEB BARBOSA      |27                |
|6           |MOT-0102    |RENAN SILVEIRA     |25                |
|7           |MOT-0041    |ALEXIA SIQUEIRA    |23                |
|8           |MOT-0081    |DR. VINÍCIUS SANTOS|23                |
|9           |MOT-0040    |CAUÊ SILVEIRA      |22                |
|10          |MOT-0105    |DR. VALENTIM MENDES|22                |
+------------+------------+-------------------+------------------+



### Utilização da frota por mês

In [11]:
veiculos_ativos = veiculos.filter(F.col("status") == "ATIVO").select("veiculo_id").distinct()
ativos_total = veiculos_ativos.count()

utilizacao_frota_mensal = (
    viagens_enriquecidas.select("mes", "veiculo_id").distinct()
    .join(veiculos_ativos, "veiculo_id", "inner")   # mantém só os veículos ativos
    .groupBy("mes")
    .agg(F.countDistinct("veiculo_id").alias("veiculos_com_viagem"))
    .withColumn("veiculos_ativos_total", F.lit(ativos_total))
    .withColumn("taxa_utilizacao",
                F.round(F.col("veiculos_com_viagem") / F.col("veiculos_ativos_total"), 4))
    .orderBy("mes")
)
utilizacao_frota_mensal.show(truncate=False)

+-------+-------------------+---------------------+---------------+
|mes    |veiculos_com_viagem|veiculos_ativos_total|taxa_utilizacao|
+-------+-------------------+---------------------+---------------+
|2026-04|115                |115                  |1.0            |
+-------+-------------------+---------------------+---------------+



### Tempo médio parado em geocercas (por tipo)

In [12]:
posicoes_geocercas = spark.read.parquet(f"{LAKEHOUSE}/analytics/posicoes_geocercas")

label = (
    F.when(F.col("geocerca_tipo") == "CENTRO_DISTRIBUICAO", "CD (Centro de Distribuição)")
     .when(F.col("geocerca_tipo") == "CLIENTE", "Cliente")
     .when(F.col("geocerca_tipo") == "PEDAGIO", "Pedágio")
     .when(F.col("geocerca_tipo") == "POSTO_COMBUSTIVEL", "Posto de Combustível")
     .otherwise(F.col("geocerca_tipo"))
)

visitas = (
    posicoes_geocercas
    .filter(F.col("geocerca_id").isNotNull())
    .groupBy("viagem_id", "geocerca_id", "geocerca_tipo")
    .agg(((F.max("timestamp").cast("long") - F.min("timestamp").cast("long")) / 60.0)
         .alias("minutos_parado"))
)

dwell = (
    visitas.groupBy("geocerca_tipo")
    .agg(
        F.count("*").alias("total_visitas"),
        F.round(F.avg("minutos_parado"), 2).alias("tempo_medio_parado_minutos"),
    )
)

todos_tipos = geocercas.select(F.col("tipo").alias("geocerca_tipo")).distinct()

tempo_parado_geocercas = (
    todos_tipos.join(dwell, "geocerca_tipo", "left")
    .withColumn("total_visitas", F.coalesce(F.col("total_visitas"), F.lit(0)))
    .withColumn("geocerca_tipo_label", label)
    .select("geocerca_tipo", "geocerca_tipo_label",
            "total_visitas", "tempo_medio_parado_minutos")
    .orderBy(F.desc("total_visitas"))
)
tempo_parado_geocercas.show(truncate=False)

+-------------------+---------------------------+-------------+--------------------------+
|geocerca_tipo      |geocerca_tipo_label        |total_visitas|tempo_medio_parado_minutos|
+-------------------+---------------------------+-------------+--------------------------+
|CENTRO_DISTRIBUICAO|CD (Centro de Distribuição)|114          |0.6                       |
|CLIENTE            |Cliente                    |29           |21.52                     |
|PEDAGIO            |Pedágio                    |0            |NULL                      |
|POSTO_COMBUSTIVEL  |Posto de Combustível       |0            |NULL                      |
+-------------------+---------------------------+-------------+--------------------------+

